In [ ]:
import os
from pdfminer.high_level import extract_text
from pdf2image import convert_from_path
import pytesseract
from PIL import Image
import json

def extract_text_from_pdf(pdf_path, output_dir="outputs", ocr_fallback=True):
    os.makedirs(output_dir, exist_ok=True)
    text_output_path = os.path.join(output_dir, os.path.basename(pdf_path).replace(".pdf", ".txt"))
    json_output_path = os.path.join(output_dir, os.path.basename(pdf_path).replace(".pdf", ".json"))

    try:
        # Try native text extraction
        text = extract_text(pdf_path)
        if not text.strip() and ocr_fallback:
            print(f"[INFO] No text layer found, using OCR for {pdf_path}...")
            images = convert_from_path(pdf_path)
            text = "\n".join(pytesseract.image_to_string(img) for img in images)
    except Exception as e:
        print(f"[ERROR] Extraction failed for {pdf_path}: {e}")
        return

    # Save plain text
    with open(text_output_path, "w", encoding="utf-8") as f:
        f.write(text)

    # Save structured JSON
    data = {"file": os.path.basename(pdf_path), "content": text}
    with open(json_output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"[SUCCESS] Extracted text saved: {text_output_path}")
    return data

    
if __name__ == "__main__":
    pdf = "sample.pdf"  # Replace with your file
    extract_text_from_pdf(pdf)
